In [16]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df = pd.read_csv("grid_load_clean1.csv")

In [6]:
df.isnull().sum()

Record_ID               0
Region                  0
Hour                    0
DayOfWeek               0
Temperature_C           0
Humidity_pct            0
Rainfall_mm             0
PopulationIndex         0
IndustrialIndex         0
SolarGenerationIndex    0
GridLoad_MW             0
dtype: int64

In [7]:
df.head()

,Record_ID,Region,Hour,DayOfWeek,Temperature_C,Humidity_pct,Rainfall_mm,PopulationIndex,IndustrialIndex,SolarGenerationIndex,GridLoad_MW
0,1,Central,20.0,0,26.7,52.0,3.1,93.0,89.0,0.03,850.3
1,2,Eastern,13.0,0,18.5,63.0,3.7,95.0,89.0,0.38,788.3
2,3,Western,14.0,4,22.7,90.0,3.3,71.0,55.0,0.13,548.0
3,4,Northern,10.0,0,19.6,41.0,0.1,75.0,78.0,0.45,604.6
4,5,Northern,12.0,0,27.4,88.0,0.0,86.0,58.0,0.70,546.1


In [8]:
df.columns

Index(['Record_ID', 'Region', 'Hour', 'DayOfWeek', 'Temperature_C',
       'Humidity_pct', 'Rainfall_mm', 'PopulationIndex', 'IndustrialIndex',
       'SolarGenerationIndex', 'GridLoad_MW'],
      dtype='str')

In [9]:
df_processed = df.copy()

In [10]:
df_processed = df_processed.drop("Record_ID", axis=1)

In [11]:
df_processed.columns

Index(['Region', 'Hour', 'DayOfWeek', 'Temperature_C', 'Humidity_pct',
       'Rainfall_mm', 'PopulationIndex', 'IndustrialIndex',
       'SolarGenerationIndex', 'GridLoad_MW'],
      dtype='str')

In [13]:
# convert the hour to morning and evening peak hours beacuase thats when the load is high and we want to predict the load during those hours
df_processed["Morning_Peak"] = df_processed["Hour"].apply(
    lambda x: 1 if x in [6,7,8,9] else 0
)

df_processed["Evening_Peak"] = df_processed["Hour"].apply(
    lambda x: 1 if x in [17,18,19,20] else 0
)

In [14]:
# create a new column to indicate if the day is a weekend or not
df_processed["Is_Weekend"] = df_processed["DayOfWeek"].apply(
    lambda x: 1 if x in [5,6] else 0
)

df_processed["Is_Working_Day"] = df_processed["DayOfWeek"].apply(
    lambda x: 1 if x < 5 else 0
)

In [15]:
df_processed.head()

,Region,Hour,DayOfWeek,Temperature_C,Humidity_pct,Rainfall_mm,PopulationIndex,IndustrialIndex,SolarGenerationIndex,GridLoad_MW,Morning_Peak,Evening_Peak,Is_Weekend,Is_Working_Day
0,Central,20.0,0,26.7,52.0,3.1,93.0,89.0,0.03,850.3,0,1,0,1
1,Eastern,13.0,0,18.5,63.0,3.7,95.0,89.0,0.38,788.3,0,0,0,1
2,Western,14.0,4,22.7,90.0,3.3,71.0,55.0,0.13,548.0,0,0,0,1
3,Northern,10.0,0,19.6,41.0,0.1,75.0,78.0,0.45,604.6,0,0,0,1
4,Northern,12.0,0,27.4,88.0,0.0,86.0,58.0,0.70,546.1,0,0,0,1


In [17]:
# Encoded categorical variables (text → numbers)

label_encoder = {}

categorical_columns = ['Region']

for col in categorical_columns:
    le = LabelEncoder()
    df_processed[col + '_Encoded'] = le.fit_transform(df_processed[col])
    label_encoder[col] = le
    
    print(f'Encoded {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')


Encoded Region: {'Central': np.int64(0), 'Eastern': np.int64(1), 'Northern': np.int64(2), 'Western': np.int64(3)}


In [18]:
df_processed.head()

,Region,Hour,DayOfWeek,Temperature_C,Humidity_pct,Rainfall_mm,PopulationIndex,IndustrialIndex,SolarGenerationIndex,GridLoad_MW,Morning_Peak,Evening_Peak,Is_Weekend,Is_Working_Day,Region_Encoded
0,Central,20.0,0,26.7,52.0,3.1,93.0,89.0,0.03,850.3,0,1,0,1,0
1,Eastern,13.0,0,18.5,63.0,3.7,95.0,89.0,0.38,788.3,0,0,0,1,1
2,Western,14.0,4,22.7,90.0,3.3,71.0,55.0,0.13,548.0,0,0,0,1,3
3,Northern,10.0,0,19.6,41.0,0.1,75.0,78.0,0.45,604.6,0,0,0,1,2
4,Northern,12.0,0,27.4,88.0,0.0,86.0,58.0,0.70,546.1,0,0,0,1,2
